# 03 — Case Study: "What Behind the Words Reaches Joey?"

Friends s02e18 Scene 9 — "The One Where Dr. Ramoray Dies."  Joey has just learned his soap-opera character is being killed off; his five friends gather at his apartment to comfort him.

Notebook 2 ended on a structural finding: *the comforter's lexical trajectory becomes dynamically indistinguishable from the protagonist's*.  This notebook makes that visible at the smallest possible scale — 24 utterances of a single scene — and reproduces Figure 4 of the report.

**Reading order:**  `01_data_and_vad` → `02_csd_detection_and_validation` → `03_case_study_scene9`

## Setup

In [ ]:
import os, sys
import pandas as pd
NB_DIR = os.path.abspath('.')
sys.path.insert(0, os.path.join(NB_DIR, '..', 'code', 'model'))

from IPython.display import Image

## 1. The 24 utterances of Scene 9

The case-study CSV was generated by `code/export_final_database.py` and contains the per-utterance VAD scores plus speaker and emotion-label columns.

In [ ]:
scene = pd.read_csv('../final_database/scene9_s02e18_with_vad.csv')
print(f'Scene 9 contains {len(scene)} utterances by '
      f'{scene["speaker"].nunique()} speakers.')
scene[['scene_position', 'speaker', 'emotion_label', 'transcript']].head(10)

## 2. The mirror problem at lexicon level

Joey is the emotional protagonist; everyone else is a comforter.  But at the lexicon level, the role assignment looks **inverted** — Joey's `Sad` lines often score *positive* (the words `feel`, `like`, `greatest` carry positive valence even in despair contexts) while comforters' `Peaceful`/`Powerful` lines often score *negative* (the words `sorry`, `worried`, `death` are negative even in comforting contexts).

In [ ]:
joey   = scene[scene['speaker'] == 'Joey Tribbiani']
others = scene[scene['speaker'] != 'Joey Tribbiani']

print('Joey utterances (Sad/Mad — the actual protagonist):')
print(joey[['transcript', 'emotion_label', 'sep']].to_string(index=False))
print()
print('Comforters (Peaceful/Powerful/Joyful):')
print(others[['speaker', 'emotion_label', 'sep']].to_string(index=False))

### Aggregate comparison

In [ ]:
joey_mean   = joey['sep'].dropna().mean()
others_mean = others['sep'].dropna().mean()

print(f'Mean SEP — Joey      (sufferer)   : {joey_mean:+.3f}')
print(f'Mean SEP — Others    (comforters) : {others_mean:+.3f}')
print(f'Lexicon-side difference            : {others_mean - joey_mean:+.3f}')
print()
print('If the lexicon were doing the job we ask of it, comforters')
print('should score ~+0.5 (Peaceful/Powerful) while Joey should')
print('score ~-0.5 (Sad).  The actual gap of ~0.15 is the lexical')
print('symptom of the role-confusion problem.')

## 3. The visualisation  ·  Figure 4

Two co-flowing rivers plus the per-utterance VAD reading.  Joey's river uses a teal collapse gradient that lightens where the surrounding comforter SEP is most positive — visualising the *failed emotional hedge*: the comforters' words rise above zero while Joey's trajectory stays flat at −0.6.  The lexicon's VAD reading (white crosses) clusters near the centre, unable to separate sufferer from comforter.

In [ ]:
%run ../code/figures/build_fig4_what_behind_the_words.py
Image('../figures/fig4_what_behind_the_words.png')

## 4. What this means for AI dialogue evaluation

Modern large language models exhibit strong **empathic mirroring**: when a user expresses distress, the system's reply tracks the user's affective trajectory — its lexical valence and dominance descend alongside the user's.  Applying single-speaker emotion-dynamics indicators ($r_1$, variance, SEP descent) unmodified to AI dialogue would render an LLM's empathic fluctuation **mathematically indistinguishable from a breakdown in the system itself** — the same emotion-attribution error documented for Phoebe in this corpus, transposed from a fictional comforter to an artificial one.

The diagnostic question is not whether the model's lexical signal descends with the user's.  It is whether the **coupled system reverses** as the user regulates back to baseline (healthy empathic engagement), or whether the model remains in a low-valence basin of its own after the user has recovered (genuine system failure).

The CSD framework, properly extended to coupled rather than single-speaker dynamics, may then serve as an early-warning device for distinguishing an AI that is *understanding* the user's pain from an AI that is itself entering an irrational tipping point.

## 5. Conclusion

Critical Slowing Down retains descriptive power on dialogue: per-character potential landscapes are interpretable, the detector's output is non-random, and the textbook signature ($r_1$ and variance rising before the SEP nadir) is recoverable on at least one in-episode case (Joey, S02E18).  But aggregate precision is only 30%, and the failure pattern is structural.  The single most informative error — emotion attribution — exposes a property of dialogue itself: the comforter's trajectory becomes dynamically indistinguishable from the protagonist's for the duration of a shared emotional event.

Any per-speaker affective time-series method applied to multi-party dialogue inherits this problem, including the empathic-mirroring evaluations now needed for large language models.  **The unit of analysis has to be the conversation, not the speaker.**